In [ ]:
# Notebook description

This notebook merges NSRDB station CSVs (skipping the first two rows and using the third row as header) for station codes from CYL_geo/CyL_geo.csv, saves per-station merged files to NSRDB/merged_files2, and adds a unified datetime "timestamp" column. It then filters CYL_GHI CSVs to the 2017–2019 period, imputes missing measurements from the merged NSRDB data using a defined column mapping, and writes the results to CYL_GHI/filtered_files. A final check lists CSVs containing blank or empty values.

Inputs: CYL_geo/CyL_geo.csv, NSRDB/*.csv, CYL_GHI/*.csv
Outputs: NSRDB/merged_files2/*_merged.csv, CYL_GHI/filtered_files/*_filtered.csv

In [ ]:
import os
import glob
import pandas as pd

# Mount Google Drive when running in Colab
try:
    from google.colab import drive
    drive.mount('/content/gdrive')
    base_dir = '/content/gdrive/MyDrive'
except Exception:
    base_dir = os.path.expanduser('~/MyDrive')

geo_path = os.path.join(base_dir, 'CYL_geo', 'CyL_geo.csv')
nsrdb_dir = os.path.join(base_dir, 'NSRDB')
merged_dir = os.path.join(nsrdb_dir, 'merged_files2')

if not os.path.exists(geo_path):
    raise FileNotFoundError(f'Geo file not found: {geo_path}')

os.makedirs(merged_dir, exist_ok=True)

geo_df = pd.read_csv(geo_path)
if 'station_code' not in geo_df.columns:
    raise KeyError('The file must contain a station_code column.')

station_codes = [str(code).strip() for code in geo_df['station_code'].dropna().unique() if str(code).strip()]

print(f'Found {len(station_codes)} station codes in {geo_path}')

for station_code in station_codes:
    pattern = os.path.join(nsrdb_dir, f'{station_code}_*.csv')
    files = sorted(glob.glob(pattern))

    if not files:
        print(f'No files found for {station_code}')
        continue

    frames = []
    for file_path in files:
        df = pd.read_csv(file_path, skiprows=2)
        frames.append(df)

    merged_df = pd.concat(frames, ignore_index=True, axis=0)

    output_path = os.path.join(merged_dir, f'{station_code}_merged.csv')
    merged_df.to_csv(output_path, index=False)

    print(f'Saved {len(files)} files for {station_code} -> {output_path}')

# Load the created merged files and build a timestamp column
merged_files = sorted(glob.glob(os.path.join(merged_dir, '*_merged.csv')))

for merged_file in merged_files:
    df = pd.read_csv(merged_file)

    required_cols = ['Year', 'Month', 'Day', 'Hour', 'Minute']
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        print(f'Skipping {merged_file}: missing columns {missing_cols}')
        continue

    df['timestamp'] = pd.to_datetime(
        df[['Year', 'Month', 'Day', 'Hour', 'Minute']].astype(str).agg(lambda r: '-'.join(r[['Year', 'Month', 'Day']]) + ' ' + ':'.join(r[['Hour', 'Minute']]), axis=1),
        format='%Y-%m-%d %H:%M',
        errors='coerce'
    )
    df = df.drop(columns=required_cols)

    df.to_csv(merged_file, index=False)
    print(f'Updated {merged_file} with timestamp column')


In [ ]:
# Filter CYL_GHI files to 2017-2019 and impute missing values from merged NSRDB data
cyl_ghi_dir = os.path.join(base_dir, 'CYL_GHI')
filtered_dir = os.path.join(cyl_ghi_dir, 'filtered_files')
os.makedirs(filtered_dir, exist_ok=True)

column_map = {
    'GHI': 'GHI',
    'wind_dir': 'Wind Direction',
    'humidity': 'Relative Humidity',
    'precipitation': 'Precipitable Water',
    'air_temp': 'Temperature',
    'wind_sp': 'Wind Speed',
    'sun_elev': 'Solar Zenith Angle',
}

for file_path in sorted(glob.glob(os.path.join(cyl_ghi_dir, '*.csv'))):
    df = pd.read_csv(file_path)

    df = df[(df['timestamp'] >= '2017-01-01') & (df['timestamp'] <= '2019-12-31 23:59:59')]

    station_code = os.path.splitext(os.path.basename(file_path))[0].split('_')[0]
    merged_station_path = os.path.join(merged_dir, f'{station_code}_merged.csv')

    if os.path.exists(merged_station_path):
        merged_df = pd.read_csv(merged_station_path, parse_dates=['timestamp'])
        merged_df['timestamp'] = pd.to_datetime(merged_df['timestamp'], errors='coerce')
        merged_df = merged_df.set_index('timestamp')
        df = df.set_index('timestamp')

        # Validate mapped columns exist in both dataframes and report missing ones
        missing_in_df = [c for c in column_map.keys() if c not in df.columns]
        missing_in_merged = [c for c in column_map.values() if c not in merged_df.columns]
        if missing_in_df:
            print(f"Warning: missing columns in {os.path.basename(file_path)}: {missing_in_df}")
        if missing_in_merged:
            print(f"Warning: missing columns in merged file {merged_station_path}: {missing_in_merged}")

        for source_col, merged_col in column_map.items():
            if source_col in df.columns and merged_col in merged_df.columns:
                fill_values = merged_df[merged_col].reindex(df.index)
                df[source_col] = df[source_col].mask(
                    df[source_col].isna() | (df[source_col].astype(str).str.strip() == ''),
                    fill_values
                )

        df = df.reset_index()
    else:
        print(f'No merged station file found for {station_code}: {merged_station_path}')

    filtered_path = os.path.join(filtered_dir, f'{station_code}_filtered.csv')
    df.to_csv(filtered_path, index=False)
    print(f'Filtered and saved {filtered_path} ({len(df)} rows)')

In [ ]:
# Join TCC column from Cloud_era5 files to filtered files based on timestamp and station name
cloud_era5_dir = os.path.join(base_dir, 'Cloud_era5')

for file_path in sorted(glob.glob(os.path.join(filtered_dir, '*.csv'))):
    df = pd.read_csv(file_path, parse_dates=['timestamp'])
    
    # Extract station name from filename
    station_name = os.path.splitext(os.path.basename(file_path))[0].split('_')[0]
    
    # Find matching Cloud_era5 file
    cloud_file_pattern = os.path.join(cloud_era5_dir, f'{station_name}*.csv')
    cloud_files = glob.glob(cloud_file_pattern)
    
    if cloud_files:
        cloud_df = pd.read_csv(cloud_files[0], parse_dates=['timestamp'])
        
        # Merge on timestamp
        df = pd.merge(df, cloud_df[['timestamp', 'tcc']], on='timestamp', how='left')
        
        # Save the updated file
        df.to_csv(file_path, index=False)
        print(f'Added TCC column to {os.path.basename(file_path)}')
    else:
        print(f'No Cloud_erra5 file found for {station_name}')

In [ ]:
# inspecting excistence of blank values
csv_files = sorted(glob.glob(os.path.join(filtered_path, '*.csv')))

for file_path in csv_files:
    df = pd.read_csv(file_path)

    blank_mask = df.isna() | df.astype(str).apply(lambda col: col.str.strip() == '')
    blank_columns = [col for col in df.columns if blank_mask[col].any()]

    if blank_columns:
        blank_counts = {col: int(blank_mask[col].sum()) for col in blank_columns}
        print(f"{os.path.basename(file_path)}: blank values found in {blank_counts}")
    else:
        print(f"{os.path.basename(file_path)}: no blank values found")

In [ ]:
# Add sinusoidal features for wind direction, sun elevation, sun azimuth, and time of day
import numpy as np

for file_path in sorted(glob.glob(os.path.join(filtered_dir, '*.csv'))):
    df = pd.read_csv(file_path, parse_dates=['timestamp'])

    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

    for col in ['wind_dir', 'sun_elev', 'sun_azim']:
        if col in df.columns:
            radians = np.radians(df[col].astype(float))
            df[f'{col}_sin'] = np.sin(radians)
            df[f'{col}_cos'] = np.cos(radians)

    if 'timestamp' in df.columns:
        seconds = (
            df['timestamp'].dt.hour * 3600
            + df['timestamp'].dt.minute * 60
            + df['timestamp'].dt.second
        )
        tod_radians = 2 * np.pi * seconds / 86400
        df['tod_sin'] = np.sin(tod_radians)
        df['tod_cos'] = np.cos(tod_radians)

    df.to_csv(file_path, index=False)
    print(f'Updated {os.path.basename(file_path)} with sinusoidal features')

In [ ]:
# --------------------------------------------------------
# Toggle whether to reuse an existing .npz tensor or rebuild it
# --------------------------------------------------------
load_existing_npz = False  # Set to True to load an existing .npz file, False to rebuild from filtered CSVs

# --------------------------------------------------------
# Output path
# --------------------------------------------------------
tensor_path = os.path.join(filtered_dir, "filtered_tensor.npz")

if load_existing_npz and os.path.exists(tensor_path):
    with np.load(tensor_path, allow_pickle=True) as data:
        tensor = data["tensor"]
        features = data["features"]
        min_vals = data["min_vals"]
        max_vals = data["max_vals"]
        normalized_features = data["normalized_features"]

    print(f"Loaded existing tensor from {tensor_path}")
    print(f"Tensor shape : {tensor.shape}")
    print(f"Loaded features : {features}")
else:
    # --------------------------------------------------------
    # Read station files
    # --------------------------------------------------------
    filtered_files = sorted(glob.glob(os.path.join(filtered_dir, "*.csv")))
    if not filtered_files:
        raise FileNotFoundError(f"No filtered CSV files found in {filtered_dir}")

    # --------------------------------------------------------
    # Fields to normalize
    # --------------------------------------------------------
    desired_features = [
        "GHI",
        "humidity",
        "precipitation",
        "air_temp",
        "wind_sp"
    ]

    # --------------------------------------------------------
    # Read all CSVs
    # --------------------------------------------------------
    dfs = []

    reference_columns = None
    reference_length = None

    for file_path in filtered_files:

        df = pd.read_csv(file_path)

        if "timestamp" in df.columns:
            df = df.drop(columns="timestamp")

        # ----------------------------------------------------
        # Check feature consistency
        # ----------------------------------------------------
        if reference_columns is None:
            reference_columns = list(df.columns)
        elif list(df.columns) != reference_columns:
            raise ValueError(
                f"\nFeature mismatch detected.\n"
                f"File: {os.path.basename(file_path)}\n"
                f"Expected:\n{reference_columns}\n"
                f"Found:\n{list(df.columns)}"
            )

        # ----------------------------------------------------
        # Check timestep consistency
        # ----------------------------------------------------
        if reference_length is None:
            reference_length = len(df)
        elif len(df) != reference_length:
            raise ValueError(
                f"\nTimestep mismatch detected.\n"
                f"File: {os.path.basename(file_path)}\n"
                f"Expected {reference_length} rows, "
                f"found {len(df)} rows."
            )

        dfs.append(df)

    # --------------------------------------------------------
    # Verify desired features exist
    # --------------------------------------------------------
    missing = [f for f in desired_features if f not in reference_columns]

    if missing:
        raise ValueError(
            "The following desired features are missing:\n"
            f"{missing}"
        )

    # --------------------------------------------------------
    # Compute global min/max
    # --------------------------------------------------------
    min_vals = {}
    max_vals = {}

    for feature in desired_features:

        values = pd.concat(
            [df[feature].astype(float) for df in dfs],
            ignore_index=True,
        )

        min_vals[feature] = values.min()
        max_vals[feature] = values.max()

    # --------------------------------------------------------
    # Output feature names
    # Original features + normalized features
    # --------------------------------------------------------
    normalized_feature_names = [f"{f}_norm" for f in desired_features]

    output_features = (
        reference_columns +
        normalized_feature_names
    )

    # --------------------------------------------------------
    # Tensor dimensions
    # --------------------------------------------------------
    T = reference_length
    N = len(dfs)
    F_original = len(reference_columns)
    F_norm = len(desired_features)
    F_total = F_original + F_norm

    tensor = np.full(
        (T, N, F_total),
        np.nan,
        dtype=float,
    )

    # --------------------------------------------------------
    # Build tensor
    # --------------------------------------------------------
    for station_idx, df in enumerate(dfs):

        # Original features
        for feature_idx, feature in enumerate(reference_columns):

            tensor[:, station_idx, feature_idx] = pd.to_numeric(
                df[feature],
                errors="coerce",
            )

        # Normalized features
        for norm_idx, feature in enumerate(desired_features):

            arr = pd.to_numeric(
                df[feature],
                errors="coerce",
            ).to_numpy(dtype=float)

            lo = min_vals[feature]
            hi = max_vals[feature]

            if np.isfinite(lo) and np.isfinite(hi):

                if hi > lo:
                    arr = (arr - lo) / (hi - lo)
                else:
                    arr = np.zeros_like(arr)

            else:
                arr = np.full_like(arr, np.nan)

            tensor[:, station_idx, F_original + norm_idx] = arr

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------
    np.savez_compressed(
        tensor_path,
        tensor=tensor,
        features=np.array(output_features, dtype=object),
        min_vals=np.array(
            [min_vals[f] for f in desired_features],
            dtype=float,
        ),
        max_vals=np.array(
            [max_vals[f] for f in desired_features],
            dtype=float,
        ),
        normalized_features=np.array(
            desired_features,
            dtype=object,
        ),
    )

    print(f"Saved tensor to {tensor_path}")
    print(f"Tensor shape : {tensor.shape}")
    print(f"Original features   : {F_original}")
    print(f"Normalized features : {F_norm}")
    print(f"Total features      : {F_total}")
